<a href="https://colab.research.google.com/github/Pam-Pam29/Summative-Assignment---Mission-Based-Reinforcement-Learning/blob/main/Notebook/Sista_Health_Mission_Based_Reinforcement_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Sista Health — Mission-Based Reinforcement Learning

**Victoria Fakunle | ALU 2026**

Trains DQN, PPO, REINFORCE, and A2C agents to optimise response strategies for the **Sista Health** maternal health assistant.



##  Install Libraries

In [ ]:
import subprocess
subprocess.run(["pip","install","gymnasium","stable-baselines3[extra]",
                "matplotlib","pandas","numpy","sb3-contrib","-q"],check=True)
print(" All libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 5.6 MB/s eta 0:00:00
 All libraries installed!


##  Create Folder Structure

In [ ]:
import os
BASE = "/kaggle/working"
for folder in ["environment","training","models/dqn",
               "models/pg/ppo","models/pg/reinforce","results"]:
    os.makedirs(os.path.join(BASE,folder), exist_ok=True)
print(" Folders created!")

 Folders created!


##  Write custom_env.py

In [ ]:
ENV_CODE = (
    "import gymnasium as gym\n"
    "import numpy as np\n"
    "from gymnasium import spaces\n\n\n"
    "class SistaHealthEnv(gym.Env):\n"
    "    TOPICS=['FGM Complications','VVF Causes','Cultural Barriers',\n"
    "        'Early Marriage','TBA Dangers','Contraception',\n"
    "        'STIs and HIV','Antenatal Care','Postpartum Care']\n"
    "    LANGUAGES=['English','Yoruba','Pidgin']\n"
    "    DOMAINS=['Sexual Health','Maternal Health']\n"
    "    ACTIONS=['Text Response','Voice Note','Resource Link','Clarify']\n"
    "    def __init__(self,render_mode=None):\n"
    "        super().__init__()\n"
    "        self.observation_space=spaces.Box(\n"
    "            low=np.array([0,0,0,0,0],dtype=np.float32),\n"
    "            high=np.array([2,1,8,2,9],dtype=np.float32),dtype=np.float32)\n"
    "        self.action_space=spaces.Discrete(4)\n"
    "        self.render_mode=render_mode\n"
    "        self.state=None; self.step_count=0\n"
    "        self.episode_reward=0; self.last_action=None\n"
    "        self.last_reward=0; self.last_feedback=''\n"
    "    def _get_obs(self): return self.state.astype(np.float32)\n"
    "    def _get_info(self):\n"
    "        return {'language':self.LANGUAGES[int(self.state[0])],\n"
    "            'domain':self.DOMAINS[int(self.state[1])],\n"
    "            'topic':self.TOPICS[int(self.state[2])],\n"
    "            'literacy':['Low','Medium','High'][int(self.state[3])],\n"
    "            'step':int(self.state[4]),'episode_reward':self.episode_reward}\n"
    "    def reset(self,seed=None,options=None):\n"
    "        super().reset(seed=seed)\n"
    "        self.state=np.array([self.np_random.integers(0,3),\n"
    "            self.np_random.integers(0,2),self.np_random.integers(0,9),\n"
    "            self.np_random.integers(0,3),0],dtype=np.float32)\n"
    "        self.step_count=0; self.episode_reward=0\n"
    "        self.last_feedback='New user session started'\n"
    "        return self._get_obs(),self._get_info()\n"
    "    def step(self,action):\n"
    "        assert self.action_space.contains(action)\n"
    "        literacy=int(self.state[3]); language=int(self.state[0]); step=int(self.state[4])\n"
    "        reward=0; feedback=''\n"
    "        if action==0:\n"
    "            if literacy==2: reward,feedback=10,'Text excellent for high literacy. +10'\n"
    "            elif literacy==1: reward,feedback=5,'Text good for medium literacy. +5'\n"
    "            else: reward,feedback=-4,'Text poor for low literacy. -4'\n"
    "        elif action==1:\n"
    "            if literacy==0:\n"
    "                reward,feedback=14,'Voice perfect for low literacy. +14'\n"
    "                if language==2: reward,feedback=16,'Voice+Pidgin for low literacy. +16'\n"
    "            elif literacy==1:\n"
    "                reward,feedback=7,'Voice good for medium literacy. +7'\n"
    "                if language==2: reward,feedback=9,'Voice+Pidgin for medium literacy. +9'\n"
    "            else: reward,feedback=2,'Voice acceptable for high literacy. +2'\n"
    "        elif action==2: reward,feedback=3,'Resource link shared. +3'\n"
    "        elif action==3:\n"
    "            if step<3: reward,feedback=4,'Clarification helpful early. +4'\n"
    "            else: reward,feedback=-2,'Clarification stalling. -2'\n"
    "        self.state[4]=min(self.state[4]+1,9)\n"
    "        self.step_count+=1; self.last_action=action\n"
    "        self.last_reward=reward; self.last_feedback=feedback\n"
    "        self.episode_reward+=reward\n"
    "        terminated=self.step_count>=10\n"
    "        return self._get_obs(),reward,terminated,False,self._get_info()\n"
    "    def render(self): pass\n"
    "    def close(self): pass\n"
)
with open("/kaggle/working/environment/custom_env.py","w") as f:
    f.write(ENV_CODE)
print(" custom_env.py written!")

 custom_env.py written!


## Random Agent Demo (No Training)

In [ ]:
import sys, numpy as np
sys.path.insert(0,"/kaggle/working")
from environment.custom_env import SistaHealthEnv

env=SistaHealthEnv(); obs,info=env.reset(seed=42)
print("="*55)
print("  RANDOM AGENT DEMO — Sista Health")
print("="*55)
print(f'  User: {info["language"]} | {info["domain"]} | {info["topic"]}')
print(f'  Literacy: {info["literacy"]}')
print("-"*55)
total=0; done=False; step=0
while not done:
    action=env.action_space.sample()
    obs,reward,term,trunc,info=env.step(action)
    print(f"  Step {step+1:2d} | {env.ACTIONS[action]:20s} | {reward:+.0f}")
    print(f"         {env.last_feedback}")
    total+=reward; done=term or trunc; step+=1
print("-"*55)
print(f"  Total Reward: {total}")
print("\n Environment working!")
env.close()

  RANDOM AGENT DEMO — Sista Health
  User: English | Maternal Health | Contraception
  Literacy: Medium
-------------------------------------------------------
  Step  1 | Clarify              | +4
         Clarification helpful early. +4
  Step  2 | Resource Link        | +3
         Resource link shared. +3
  Step  3 | Clarify              | +4
         Clarification helpful early. +4
  Step  4 | Clarify              | -2
         Clarification stalling. -2
  Step  5 | Voice Note           | +7
         Voice good for medium literacy. +7
  Step  6 | Voice Note           | +7
         Voice good for medium literacy. +7
  Step  7 | Voice Note           | +7
         Voice good for medium literacy. +7
  Step  8 | Text Response        | +5
         Text good for medium literacy. +5
  Step  9 | Clarify              | -2
         Clarification stalling. -2
  Step 10 | Text Response        | +5
         Text good for medium literacy. +5
------------------------------------------------------

## Shared Callback & Evaluate (Run Once)

In [ ]:
import os, sys, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
sys.path.insert(0,"/kaggle/working")
from environment.custom_env import SistaHealthEnv

RESULTS_DIR = "/kaggle/working/results"
TIMESTEPS   = 100_000

class PGCallback(BaseCallback):
    def __init__(self):
        super().__init__(0)
        self.episode_rewards,self.current_rewards,self.entropy_log=[],[],[]
    def _on_step(self):
        self.current_rewards.append(self.locals["rewards"][0])
        if self.locals["dones"][0]:
            self.episode_rewards.append(sum(self.current_rewards))
            self.current_rewards=[]
        try:
            dist=self.model.policy.action_dist
            self.entropy_log.append(dist.entropy().mean().item())
        except: pass
        return True

class DQNCallback(BaseCallback):
    def __init__(self):
        super().__init__(0)
        self.episode_rewards,self.current_rewards=[],[]
        self.loss_log,self.step_log=[],[]
    def _on_step(self):
        self.current_rewards.append(self.locals["rewards"][0])
        if self.locals["dones"][0]:
            self.episode_rewards.append(sum(self.current_rewards))
            self.current_rewards=[]
        try:
            loss=self.model.logger.name_to_value.get("train/loss",None)
            if loss is not None:
                self.loss_log.append(loss); self.step_log.append(self.num_timesteps)
        except: pass
        return True

def evaluate_model(model,n=30):
    env=SistaHealthEnv(); rewards=[]
    for _ in range(n):
        obs,_=env.reset(); ep_r=0; done=False
        while not done:
            action,_=model.predict(obs,deterministic=True)
            obs,r,term,trunc,_=env.step(int(action))
            ep_r+=r; done=term or trunc
        rewards.append(ep_r)
    env.close()
    return np.mean(rewards),np.std(rewards)

print(" Callbacks and evaluate_model ready!")

2026-03-26 15:13:08.297262: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774537988.521504      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774537988.610477      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774537989.129988      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774537989.130026      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774537989.130028      55 computation_placer.cc:177] computation placer alr

 Callbacks and evaluate_model ready!


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


##  DQN Training (10 Experiments)


In [ ]:
from stable_baselines3 import DQN

DQN_EXPS = [
    # Run 1 — Baseline
    {"learning_rate":1e-3,"gamma":0.99,"batch_size":64,"buffer_size":50000,"exploration_fraction":0.3,"exploration_final_eps":0.05},
    # Run 2 — Low LR + large buffer + tight eps
    {"learning_rate":1e-4,"gamma":0.99,"batch_size":64,"buffer_size":100000,"exploration_fraction":0.3,"exploration_final_eps":0.01},
    # Run 3 — High LR + large batch + less exploration
    {"learning_rate":5e-3,"gamma":0.99,"batch_size":256,"buffer_size":50000,"exploration_fraction":0.15,"exploration_final_eps":0.02},
    # Run 4 — Low gamma + small batch + high exploration (myopic)
    {"learning_rate":1e-3,"gamma":0.85,"batch_size":32,"buffer_size":50000,"exploration_fraction":0.5,"exploration_final_eps":0.1},
    # Run 5 — High gamma + large batch + large buffer (far-sighted)
    {"learning_rate":5e-4,"gamma":0.995,"batch_size":256,"buffer_size":100000,"exploration_fraction":0.2,"exploration_final_eps":0.02},
    # Run 6 — Medium LR + large buffer + low final eps
    {"learning_rate":1e-3,"gamma":0.99,"batch_size":64,"buffer_size":100000,"exploration_fraction":0.3,"exploration_final_eps":0.01},
    # Run 7 — High LR + low gamma + small batch (aggressive short-horizon)
    {"learning_rate":2e-3,"gamma":0.97,"batch_size":32,"buffer_size":50000,"exploration_fraction":0.1,"exploration_final_eps":0.01},
    # Run 8 — Max exploration + low gamma + medium batch (noisy)
    {"learning_rate":5e-4,"gamma":0.90,"batch_size":128,"buffer_size":50000,"exploration_fraction":0.6,"exploration_final_eps":0.1},
    # Run 9 — Tuned combo: balanced LR + large buffer + medium batch
    {"learning_rate":5e-4,"gamma":0.99,"batch_size":128,"buffer_size":100000,"exploration_fraction":0.25,"exploration_final_eps":0.02},
    # Run 10 — Conservative: very low LR + large batch + high gamma
    {"learning_rate":2e-4,"gamma":0.995,"batch_size":256,"buffer_size":100000,"exploration_fraction":0.2,"exploration_final_eps":0.02},
]

dqn_results,dqn_callbacks=[],[]
dqn_best_reward=-float("inf"); dqn_best_model=None

print("="*65)
print("   DQN Experiments — Sista Health RL")
print("="*65)

for i,p in enumerate(DQN_EXPS):
    print(f"\n[DQN Run {i+1}/10]  LR={p['learning_rate']}  gamma={p['gamma']}  "
          f"batch={p['batch_size']}  buffer={p['buffer_size']}  explore={p['exploration_fraction']}")
    env=Monitor(SistaHealthEnv()); cb=DQNCallback()
    model=DQN("MlpPolicy",env,
              learning_rate=p["learning_rate"],gamma=p["gamma"],
              batch_size=p["batch_size"],buffer_size=p["buffer_size"],
              exploration_fraction=p["exploration_fraction"],
              exploration_final_eps=p["exploration_final_eps"],
              learning_starts=1000,target_update_interval=500,verbose=0)
    model.learn(total_timesteps=TIMESTEPS,callback=cb)
    mean_r,std_r=evaluate_model(model)
    print(f"   → Mean Reward: {mean_r:.2f} ± {std_r:.2f}")
    if mean_r>dqn_best_reward:
        dqn_best_reward=mean_r; dqn_best_model=model
        model.save("/kaggle/working/models/dqn/best_dqn_model")
        with open("/kaggle/working/models/dqn/best_run.txt","w") as f: f.write(str(i+1))
        print("   →  Saved as best DQN model!")
    dqn_results.append({"Run":i+1,"Learning Rate":p["learning_rate"],"Gamma":p["gamma"],
        "Batch Size":p["batch_size"],"Buffer Size":p["buffer_size"],
        "Explore Fraction":p["exploration_fraction"],"Final Eps":p["exploration_final_eps"],
        "Mean Reward":round(mean_r,2),"Std Reward":round(std_r,2)})
    dqn_callbacks.append(cb); env.close()

dqn_df=pd.DataFrame(dqn_results)
dqn_df.to_csv(f"{RESULTS_DIR}/dqn_results.csv",index=False)
print("\n"+"="*80); print("DQN RESULTS TABLE"); print("="*80)
print(dqn_df.to_string(index=False))
best_row=dqn_df.loc[dqn_df["Mean Reward"].idxmax()]
print(f"\nBest DQN Run: #{int(best_row['Run'])}  Mean: {best_row['Mean Reward']}")
print("\n DQN training complete!")

   DQN Experiments — Sista Health RL

[DQN Run 1/10]  LR=0.001  gamma=0.99  batch=64  buffer=50000  explore=0.3
   → Mean Reward: 107.63 ± 32.71
   →  Saved as best DQN model!

[DQN Run 2/10]  LR=0.0001  gamma=0.99  batch=64  buffer=100000  explore=0.3
   → Mean Reward: 30.67 ± 60.82

[DQN Run 3/10]  LR=0.005  gamma=0.99  batch=256  buffer=50000  explore=0.15
   → Mean Reward: 112.33 ± 30.62
   →  Saved as best DQN model!

[DQN Run 4/10]  LR=0.001  gamma=0.85  batch=32  buffer=50000  explore=0.5
   → Mean Reward: 112.33 ± 33.13

[DQN Run 5/10]  LR=0.0005  gamma=0.995  batch=256  buffer=100000  explore=0.2
   → Mean Reward: 105.67 ± 22.61

[DQN Run 6/10]  LR=0.001  gamma=0.99  batch=64  buffer=100000  explore=0.3
   → Mean Reward: 109.67 ± 31.12

[DQN Run 7/10]  LR=0.002  gamma=0.97  batch=32  buffer=50000  explore=0.1
   → Mean Reward: 108.67 ± 32.01

[DQN Run 8/10]  LR=0.0005  gamma=0.9  batch=128  buffer=50000  explore=0.6
   → Mean Reward: 109.67 ± 29.72

[DQN Run 9/10]  LR=0.0005  

##  DQN Plots

In [ ]:
runs=[r["Run"] for r in dqn_results]; mean_r=[r["Mean Reward"] for r in dqn_results]
std_r=[r["Std Reward"] for r in dqn_results]; best_idx=int(np.argmax(mean_r))

# Hyperparameter comparison
fig,axes=plt.subplots(2,3,figsize=(16,10))
fig.suptitle("DQN Hyperparameter Comparison",fontsize=14,fontweight="bold")
bars=axes[0,0].bar(runs,mean_r,color="#58a6ff",alpha=0.85,yerr=std_r,capsize=4)
bars[best_idx].set_color("#25d366")
axes[0,0].set_title("Mean Reward per Run"); axes[0,0].set_xlabel("Run #"); axes[0,0].set_ylabel("Mean Reward")
for ax,key,col in zip([axes[0,1],axes[0,2],axes[1,0],axes[1,1],axes[1,2]],
    ["Learning Rate","Gamma","Batch Size","Buffer Size","Explore Fraction"],
    ["#e3b341","#f85149","#a371f7","#ffa657","#39d353"]):
    xs=[r[key] for r in dqn_results]
    ax.scatter(xs,mean_r,color=col,s=80,zorder=3)
    for i,(x,y) in enumerate(zip(xs,mean_r)): ax.annotate(f"R{i+1}",(x,y),textcoords="offset points",xytext=(4,4),fontsize=7)
    ax.set_title(f"{key} vs Mean Reward"); ax.set_xlabel(key); ax.set_ylabel("Mean Reward"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/dqn_experiments.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: dqn_experiments.png")

# Reward + Cumulative curve
rews=dqn_callbacks[best_idx].episode_rewards
fig,axes=plt.subplots(1,2,figsize=(14,5)); fig.suptitle(f"DQN Reward Curve — Best Run #{best_idx+1}",fontsize=13,fontweight="bold")
axes[0].plot(rews,color="#58a6ff",alpha=0.3,linewidth=1,label="Episode Reward")
if len(rews)>10:
    w=max(10,len(rews)//20); sm=[np.mean(rews[max(0,i-w):i+1]) for i in range(len(rews))]
    axes[0].plot(sm,color="#58a6ff",linewidth=2,label=f"Smoothed (w={w})")
axes[0].set_title("Episode Reward"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
cum=np.cumsum(rews); axes[1].plot(cum,color="#25d366",linewidth=2); axes[1].fill_between(range(len(cum)),cum,alpha=0.15,color="#25d366")
axes[1].set_title("Cumulative Reward"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/dqn_reward_curve.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: dqn_reward_curve.png")

# Objective (TD loss + Q-proxy)
cb=dqn_callbacks[best_idx]
fig,axes=plt.subplots(1,2,figsize=(14,5)); fig.suptitle(f"DQN Objective Curve — Best Run #{best_idx+1}",fontsize=13,fontweight="bold")
if cb.loss_log:
    axes[0].plot(cb.step_log,cb.loss_log,color="#f85149",linewidth=1.5,label="TD Loss")
else:
    axes[0].text(0.5,0.5,"Loss not logged",ha="center",va="center",transform=axes[0].transAxes,fontsize=11,color="gray")
axes[0].set_title("TD Loss"); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)
if len(rews)>10:
    qp=[np.mean(rews[max(0,i-15):i+1]) for i in range(len(rews))]
    axes[1].plot(qp,color="#e3b341",linewidth=2,label="Q-value Proxy"); axes[1].fill_between(range(len(qp)),qp,alpha=0.15,color="#e3b341")
axes[1].set_title("Q-value Proxy"); axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/dqn_objective_curve.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: dqn_objective_curve.png")

# Convergence
fig,axes=plt.subplots(1,2,figsize=(14,5)); fig.suptitle("DQN Convergence Analysis",fontsize=13,fontweight="bold")
th=40; conv=[]
for cb2 in dqn_callbacks:
    r2=cb2.episode_rewards; ce=len(r2)
    if len(r2)>20:
        for ep in range(20,len(r2)):
            if np.mean(r2[ep-20:ep])>=th: ce=ep; break
    conv.append(ce)
clrs=["#25d366" if c<len(dqn_callbacks[0].episode_rewards) else "#f85149" for c in conv]
axes[0].bar(runs,conv,color=clrs,alpha=0.85); axes[0].axhline(th,color="gold",linestyle="--",linewidth=1.5,label=f"Threshold:{th}")
axes[0].set_title(f"Episodes to Converge (≥{th})"); axes[0].legend(fontsize=8)
fm=[np.mean(cb2.episode_rewards[int(len(cb2.episode_rewards)*0.8):]) if cb2.episode_rewards else 0 for cb2 in dqn_callbacks]
fs=[np.std( cb2.episode_rewards[int(len(cb2.episode_rewards)*0.8):]) if cb2.episode_rewards else 0 for cb2 in dqn_callbacks]
axes[1].errorbar(runs,fm,yerr=fs,fmt="o-",color="#58a6ff",capsize=5,linewidth=2,markersize=7)
axes[1].set_title("Final Stability (Last 20%)"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/dqn_convergence.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: dqn_convergence.png")

# Generalization heatmap
env3=SistaHealthEnv(); cases=[]
for lang in range(3):
    for lit in range(3):
        rws=[]
        for _ in range(10):
            ob,_=env3.reset(); ob[0]=lang; ob[3]=lit; ob[4]=0
            ep=0; dn=False; st=0
            while not dn and st<10:
                ac,_=dqn_best_model.predict(ob,deterministic=True)
                ob,rw,tr,tc,_=env3.step(int(ac)); ep+=rw; dn=tr or tc; st+=1
            rws.append(ep)
        cases.append({"label":f"{'EYP'[lang]}-{'LMH'[lit]}","lang":lang,"lit":lit,"mean":np.mean(rws)})
env3.close()
fig,axes=plt.subplots(1,2,figsize=(14,6)); fig.suptitle("DQN Generalization Test",fontsize=13,fontweight="bold")
lbs=[c["label"] for c in cases]; ms=[c["mean"] for c in cases]
clrs2=["#25d366" if m>0 else "#f85149" for m in ms]
axes[0].bar(range(len(lbs)),ms,color=clrs2,alpha=0.85); axes[0].set_xticks(range(len(lbs)))
axes[0].set_xticklabels(lbs,rotation=45,ha="right",fontsize=9)
axes[0].set_title("Mean Reward by Profile (E/Y/P=lang, L/M/H=lit)"); axes[0].set_ylabel("Mean Reward")
grid=np.zeros((3,3))
for c in cases: grid[c["lit"],c["lang"]]=c["mean"]
im=axes[1].imshow(grid,cmap="RdYlGn",aspect="auto"); plt.colorbar(im,ax=axes[1])
axes[1].set_xticks([0,1,2]); axes[1].set_xticklabels(["English","Yoruba","Pidgin"])
axes[1].set_yticks([0,1,2]); axes[1].set_yticklabels(["Low","Medium","High"])
axes[1].set_title("Reward Heatmap")
for i in range(3):
    for j in range(3): axes[1].text(j,i,f"{grid[i,j]:.0f}",ha="center",va="center",fontsize=10,fontweight="bold")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/dqn_generalization.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: dqn_generalization.png")
print("\n All DQN plots done!")

Saved: dqn_experiments.png
Saved: dqn_reward_curve.png
Saved: dqn_objective_curve.png
Saved: dqn_convergence.png
Saved: dqn_generalization.png

 All DQN plots done!


##  PPO Training (10 Experiments)


In [ ]:
from stable_baselines3 import PPO

PPO_EXPS = [
    # Run 1 — Baseline
    {"learning_rate":3e-4,"gamma":0.99,"n_steps":2048,"ent_coef":0.01,"clip_range":0.2,"gae_lambda":0.95},
    # Run 2 — Low LR + short rollout + low entropy (conservative)
    {"learning_rate":1e-4,"gamma":0.99,"n_steps":512,"ent_coef":0.005,"clip_range":0.15,"gae_lambda":0.92},
    # Run 3 — High LR + long rollout + high entropy (aggressive)
    {"learning_rate":1e-3,"gamma":0.99,"n_steps":4096,"ent_coef":0.05,"clip_range":0.2,"gae_lambda":0.98},
    # Run 4 — Low gamma + short rollout (myopic)
    {"learning_rate":3e-4,"gamma":0.85,"n_steps":512,"ent_coef":0.01,"clip_range":0.2,"gae_lambda":0.90},
    # Run 5 — High gamma + wide clip (far-sighted)
    {"learning_rate":3e-4,"gamma":0.995,"n_steps":2048,"ent_coef":0.01,"clip_range":0.3,"gae_lambda":0.98},
    # Run 6 — Zero entropy + tight clip (pure exploitation)
    {"learning_rate":5e-4,"gamma":0.99,"n_steps":1024,"ent_coef":0.0,"clip_range":0.1,"gae_lambda":0.95},
    # Run 7 — High entropy + low gamma + long rollout (curious short-sighted)
    {"learning_rate":2e-4,"gamma":0.90,"n_steps":4096,"ent_coef":0.05,"clip_range":0.25,"gae_lambda":0.92},
    # Run 8 — Medium LR + medium rollout + tight clip (stable balanced)
    {"learning_rate":3e-4,"gamma":0.97,"n_steps":1024,"ent_coef":0.02,"clip_range":0.15,"gae_lambda":0.95},
    # Run 9 — Very low LR + very long rollout + high gae_lambda
    {"learning_rate":5e-5,"gamma":0.99,"n_steps":4096,"ent_coef":0.01,"clip_range":0.2,"gae_lambda":0.99},
    # Run 10 — Tuned: moderate LR + balanced rollout + small entropy
    {"learning_rate":2e-4,"gamma":0.995,"n_steps":2048,"ent_coef":0.02,"clip_range":0.25,"gae_lambda":0.98},
]

ppo_results,ppo_callbacks=[],[]
ppo_best_reward=-float("inf"); ppo_best_model=None

print("="*65); print("   PPO Experiments — Sista Health RL"); print("="*65)

for i,p in enumerate(PPO_EXPS):
    print(f"\n[PPO Run {i+1}/10]  LR={p['learning_rate']}  gamma={p['gamma']}  "
          f"n_steps={p['n_steps']}  ent={p['ent_coef']}  clip={p['clip_range']}")
    env=Monitor(SistaHealthEnv()); cb=PGCallback()
    model=PPO("MlpPolicy",env,learning_rate=p["learning_rate"],gamma=p["gamma"],
              n_steps=p["n_steps"],ent_coef=p["ent_coef"],
              clip_range=p["clip_range"],gae_lambda=p["gae_lambda"],verbose=0)
    model.learn(total_timesteps=TIMESTEPS,callback=cb)
    mean_r,std_r=evaluate_model(model)
    print(f"   → Mean Reward: {mean_r:.2f} ± {std_r:.2f}")
    if mean_r>ppo_best_reward:
        ppo_best_reward=mean_r; ppo_best_model=model
        model.save("/kaggle/working/models/pg/ppo/best_ppo_model")
        with open("/kaggle/working/models/pg/ppo/best_run.txt","w") as f: f.write(str(i+1))
        print("   →  Saved as best PPO model!")
    ppo_results.append({"Run":i+1,"Learning Rate":p["learning_rate"],"Gamma":p["gamma"],
        "N Steps":p["n_steps"],"Entropy Coef":p["ent_coef"],"Clip Range":p["clip_range"],
        "GAE Lambda":p["gae_lambda"],"Mean Reward":round(mean_r,2),"Std Reward":round(std_r,2)})
    ppo_callbacks.append(cb); env.close()

ppo_df=pd.DataFrame(ppo_results)
ppo_df.to_csv(f"{RESULTS_DIR}/ppo_results.csv",index=False)
print("\n── PPO Results ──"); print(ppo_df.to_string(index=False))
print("\n PPO training complete!")

   PPO Experiments — Sista Health RL

[PPO Run 1/10]  LR=0.0003  gamma=0.99  n_steps=2048  ent=0.01  clip=0.2


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


   → Mean Reward: 114.33 ± 28.60
   →  Saved as best PPO model!

[PPO Run 2/10]  LR=0.0001  gamma=0.99  n_steps=512  ent=0.005  clip=0.15
   → Mean Reward: 103.80 ± 34.22

[PPO Run 3/10]  LR=0.001  gamma=0.99  n_steps=4096  ent=0.05  clip=0.2
   → Mean Reward: 103.40 ± 33.44

[PPO Run 4/10]  LR=0.0003  gamma=0.85  n_steps=512  ent=0.01  clip=0.2
   → Mean Reward: 104.67 ± 31.91

[PPO Run 5/10]  LR=0.0003  gamma=0.995  n_steps=2048  ent=0.01  clip=0.3
   → Mean Reward: 107.67 ± 28.01

[PPO Run 6/10]  LR=0.0005  gamma=0.99  n_steps=1024  ent=0.0  clip=0.1
   → Mean Reward: 102.07 ± 28.28

[PPO Run 7/10]  LR=0.0002  gamma=0.9  n_steps=4096  ent=0.05  clip=0.25
   → Mean Reward: 106.13 ± 27.10

[PPO Run 8/10]  LR=0.0003  gamma=0.97  n_steps=1024  ent=0.02  clip=0.15
   → Mean Reward: 107.33 ± 29.66

[PPO Run 9/10]  LR=5e-05  gamma=0.99  n_steps=4096  ent=0.01  clip=0.2
   → Mean Reward: 106.80 ± 31.06

[PPO Run 10/10]  LR=0.0002  gamma=0.995  n_steps=2048  ent=0.02  clip=0.25
   → Mean Rew

##  PPO Plots

In [ ]:
runs=[r["Run"] for r in ppo_results]; mean_r=[r["Mean Reward"] for r in ppo_results]
std_r=[r["Std Reward"] for r in ppo_results]; best_idx=int(np.argmax(mean_r))

fig,axes=plt.subplots(2,3,figsize=(16,10))
fig.suptitle("PPO Hyperparameter Comparison",fontsize=14,fontweight="bold")
bars=axes[0,0].bar(runs,mean_r,color="#25d366",alpha=0.85,yerr=std_r,capsize=4); bars[best_idx].set_color("#58a6ff")
axes[0,0].set_title("Mean Reward per Run")
for ax,key,col in zip([axes[0,1],axes[0,2],axes[1,0],axes[1,1],axes[1,2]],
    ["Learning Rate","N Steps","Gamma","Entropy Coef","Clip Range"],
    ["#e3b341","#f85149","#a371f7","#ffa657","#39d353"]):
    xs=[r[key] for r in ppo_results]
    ax.scatter(xs,mean_r,color=col,s=80,zorder=3)
    for i,(x,y) in enumerate(zip(xs,mean_r)): ax.annotate(f"R{i+1}",(x,y),textcoords="offset points",xytext=(4,4),fontsize=7)
    ax.set_title(f"{key} vs Mean Reward"); ax.set_xlabel(key); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/ppo_experiments.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: ppo_experiments.png")

rews=ppo_callbacks[best_idx].episode_rewards
fig,axes=plt.subplots(1,2,figsize=(14,5)); fig.suptitle(f"PPO Reward Curve — Best Run #{best_idx+1}",fontsize=13,fontweight="bold")
axes[0].plot(rews,color="#25d366",alpha=0.3,linewidth=1,label="Episode Reward")
if len(rews)>10:
    w=max(10,len(rews)//20); sm=[np.mean(rews[max(0,i-w):i+1]) for i in range(len(rews))]
    axes[0].plot(sm,color="#25d366",linewidth=2,label="Smoothed")
axes[0].set_title("Episode Reward"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
cum=np.cumsum(rews); axes[1].plot(cum,color="#25d366",linewidth=2); axes[1].fill_between(range(len(cum)),cum,alpha=0.15,color="#25d366")
axes[1].set_title("Cumulative Reward"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/ppo_reward_curve.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: ppo_reward_curve.png")

cb=ppo_callbacks[best_idx]
fig,axes=plt.subplots(1,2,figsize=(14,5)); fig.suptitle("PPO Entropy Curves",fontsize=13,fontweight="bold")
if cb.entropy_log:
    xs=np.linspace(0,1,len(cb.entropy_log)); axes[0].plot(xs,cb.entropy_log,color="#25d366",linewidth=1.5,label="Entropy")
    axes[0].fill_between(xs,cb.entropy_log,alpha=0.2,color="#25d366")
else:
    axes[0].text(0.5,0.5,"Entropy not captured",ha="center",va="center",transform=axes[0].transAxes,fontsize=10,color="gray")
axes[0].set_title("Policy Entropy (direct)"); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)
if len(rews)>20:
    w=20; sp=[np.std(rews[max(0,i-w):i+1]) for i in range(len(rews))]
    xs=np.linspace(0,1,len(sp)); axes[1].plot(xs,sp,color="#e3b341",linewidth=1.5,label="Rolling Reward Std")
    axes[1].fill_between(xs,sp,alpha=0.2,color="#e3b341")
axes[1].set_title("Entropy Proxy (Rolling Std)"); axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/ppo_entropy_curve.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: ppo_entropy_curve.png")
print("\n All PPO plots done!")

Saved: ppo_experiments.png
Saved: ppo_reward_curve.png
Saved: ppo_entropy_curve.png

 All PPO plots done!


##  REINFORCE Training (10 Experiments)


In [ ]:
from stable_baselines3 import A2C

RF_EXPS = [
    # Run 1 — Baseline
    {"learning_rate":7e-4,"gamma":0.99,"n_steps":20,"ent_coef":0.01,"vf_coef":0.0,"max_grad_norm":0.5},
    # Run 2 — Low LR + long rollout + no entropy
    {"learning_rate":1e-4,"gamma":0.99,"n_steps":50,"ent_coef":0.0,"vf_coef":0.0,"max_grad_norm":0.5},
    # Run 3 — High LR + short rollout + high entropy (fast + noisy)
    {"learning_rate":1e-3,"gamma":0.99,"n_steps":10,"ent_coef":0.05,"vf_coef":0.0,"max_grad_norm":1.0},
    # Run 4 — Low gamma + long rollout (myopic)
    {"learning_rate":7e-4,"gamma":0.85,"n_steps":50,"ent_coef":0.01,"vf_coef":0.0,"max_grad_norm":0.5},
    # Run 5 — High gamma + short rollout + with baseline
    {"learning_rate":5e-4,"gamma":0.995,"n_steps":10,"ent_coef":0.01,"vf_coef":0.25,"max_grad_norm":0.5},
    # Run 6 — Medium LR + medium rollout + strong baseline
    {"learning_rate":3e-4,"gamma":0.99,"n_steps":30,"ent_coef":0.02,"vf_coef":0.5,"max_grad_norm":0.5},
    # Run 7 — High LR + long rollout + strong grad clip
    {"learning_rate":1e-3,"gamma":0.97,"n_steps":50,"ent_coef":0.01,"vf_coef":0.1,"max_grad_norm":0.3},
    # Run 8 — Low gamma + high entropy + no baseline (unstable)
    {"learning_rate":7e-4,"gamma":0.90,"n_steps":20,"ent_coef":0.05,"vf_coef":0.0,"max_grad_norm":1.0},
    # Run 9 — Very low LR + full episode + small baseline (careful)
    {"learning_rate":5e-5,"gamma":0.99,"n_steps":50,"ent_coef":0.005,"vf_coef":0.1,"max_grad_norm":0.5},
    # Run 10 — Best combo: moderate LR + full episode + small entropy + baseline
    {"learning_rate":5e-4,"gamma":0.995,"n_steps":50,"ent_coef":0.02,"vf_coef":0.1,"max_grad_norm":0.5},
]

rf_results,rf_callbacks=[],[]
rf_best_reward=-float("inf"); rf_best_model=None

print("="*65); print("   REINFORCE Experiments — Sista Health RL"); print("="*65)

for i,p in enumerate(RF_EXPS):
    print(f"\n[REINFORCE Run {i+1}/10]  LR={p['learning_rate']}  gamma={p['gamma']}  "
          f"n_steps={p['n_steps']}  ent={p['ent_coef']}  vf={p['vf_coef']}")
    env=Monitor(SistaHealthEnv()); cb=PGCallback()
    model=A2C("MlpPolicy",env,learning_rate=p["learning_rate"],gamma=p["gamma"],
              n_steps=p["n_steps"],ent_coef=p["ent_coef"],
              vf_coef=p["vf_coef"],max_grad_norm=p["max_grad_norm"],verbose=0)
    model.learn(total_timesteps=TIMESTEPS,callback=cb)
    mean_r,std_r=evaluate_model(model)
    print(f"   → Mean Reward: {mean_r:.2f} ± {std_r:.2f}")
    if mean_r>rf_best_reward:
        rf_best_reward=mean_r; rf_best_model=model
        model.save("/kaggle/working/models/pg/reinforce/best_reinforce_model")
        with open("/kaggle/working/models/pg/reinforce/best_run.txt","w") as f: f.write(str(i+1))
        print("   →  Saved as best REINFORCE model!")
    rf_results.append({"Run":i+1,"Learning Rate":p["learning_rate"],"Gamma":p["gamma"],
        "N Steps":p["n_steps"],"Entropy Coef":p["ent_coef"],"VF Coef":p["vf_coef"],
        "Max Grad Norm":p["max_grad_norm"],"Mean Reward":round(mean_r,2),"Std Reward":round(std_r,2)})
    rf_callbacks.append(cb); env.close()

rf_df=pd.DataFrame(rf_results)
rf_df.to_csv(f"{RESULTS_DIR}/reinforce_results.csv",index=False)
print("\n── REINFORCE Results ──"); print(rf_df.to_string(index=False))
print("\n REINFORCE training complete!")

   REINFORCE Experiments — Sista Health RL

[REINFORCE Run 1/10]  LR=0.0007  gamma=0.99  n_steps=20  ent=0.01  vf=0.0


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


   → Mean Reward: 72.33 ± 48.49
   →  Saved as best REINFORCE model!

[REINFORCE Run 2/10]  LR=0.0001  gamma=0.99  n_steps=50  ent=0.0  vf=0.0
   → Mean Reward: 77.33 ± 48.78
   →  Saved as best REINFORCE model!

[REINFORCE Run 3/10]  LR=0.001  gamma=0.99  n_steps=10  ent=0.05  vf=0.0
   → Mean Reward: 79.67 ± 54.07
   →  Saved as best REINFORCE model!

[REINFORCE Run 4/10]  LR=0.0007  gamma=0.85  n_steps=50  ent=0.01  vf=0.0
   → Mean Reward: 86.33 ± 45.72
   →  Saved as best REINFORCE model!

[REINFORCE Run 5/10]  LR=0.0005  gamma=0.995  n_steps=10  ent=0.01  vf=0.25
   → Mean Reward: 100.93 ± 26.26
   →  Saved as best REINFORCE model!

[REINFORCE Run 6/10]  LR=0.0003  gamma=0.99  n_steps=30  ent=0.02  vf=0.5
   → Mean Reward: 103.47 ± 29.85
   →  Saved as best REINFORCE model!

[REINFORCE Run 7/10]  LR=0.001  gamma=0.97  n_steps=50  ent=0.01  vf=0.1
   → Mean Reward: 84.00 ± 50.57

[REINFORCE Run 8/10]  LR=0.0007  gamma=0.9  n_steps=20  ent=0.05  vf=0.0
   → Mean Reward: 77.00 ± 52.

## REINFORCE Plots

In [ ]:
runs=[r["Run"] for r in rf_results]; mean_r=[r["Mean Reward"] for r in rf_results]
std_r=[r["Std Reward"] for r in rf_results]; best_idx=int(np.argmax(mean_r))

fig,axes=plt.subplots(2,3,figsize=(16,10))
fig.suptitle("REINFORCE Hyperparameter Comparison",fontsize=14,fontweight="bold")
bars=axes[0,0].bar(runs,mean_r,color="#e3b341",alpha=0.85,yerr=std_r,capsize=4); bars[best_idx].set_color("#58a6ff")
axes[0,0].set_title("Mean Reward per Run")
for ax,key,col in zip([axes[0,1],axes[0,2],axes[1,0],axes[1,1],axes[1,2]],
    ["Learning Rate","N Steps","Gamma","Entropy Coef","VF Coef"],
    ["#58a6ff","#f85149","#a371f7","#ffa657","#39d353"]):
    xs=[r[key] for r in rf_results]
    ax.scatter(xs,mean_r,color=col,s=80,zorder=3)
    for i,(x,y) in enumerate(zip(xs,mean_r)): ax.annotate(f"R{i+1}",(x,y),textcoords="offset points",xytext=(4,4),fontsize=7)
    ax.set_title(f"{key} vs Mean Reward"); ax.set_xlabel(key); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/reinforce_experiments.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: reinforce_experiments.png")

rews=rf_callbacks[best_idx].episode_rewards
fig,axes=plt.subplots(1,2,figsize=(14,5)); fig.suptitle(f"REINFORCE Reward Curve — Best Run #{best_idx+1}",fontsize=13,fontweight="bold")
axes[0].plot(rews,color="#e3b341",alpha=0.3,linewidth=1,label="Episode Reward")
if len(rews)>10:
    w=max(10,len(rews)//20); sm=[np.mean(rews[max(0,i-w):i+1]) for i in range(len(rews))]
    axes[0].plot(sm,color="#e3b341",linewidth=2,label="Smoothed")
axes[0].set_title("Episode Reward"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
cum=np.cumsum(rews); axes[1].plot(cum,color="#e3b341",linewidth=2); axes[1].fill_between(range(len(cum)),cum,alpha=0.15,color="#e3b341")
axes[1].set_title("Cumulative Reward"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/reinforce_reward_curve.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: reinforce_reward_curve.png")

cb=rf_callbacks[best_idx]
fig,axes=plt.subplots(1,2,figsize=(14,5)); fig.suptitle("REINFORCE Entropy Curves",fontsize=13,fontweight="bold")
if cb.entropy_log:
    xs=np.linspace(0,1,len(cb.entropy_log)); axes[0].plot(xs,cb.entropy_log,color="#e3b341",linewidth=1.5,label="Entropy")
    axes[0].fill_between(xs,cb.entropy_log,alpha=0.2,color="#e3b341")
else:
    axes[0].text(0.5,0.5,"Entropy not captured",ha="center",va="center",transform=axes[0].transAxes,fontsize=10,color="gray")
axes[0].set_title("Policy Entropy"); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)
if len(rews)>20:
    w=20; sp=[np.std(rews[max(0,i-w):i+1]) for i in range(len(rews))]
    xs=np.linspace(0,1,len(sp)); axes[1].plot(xs,sp,color="#ffa657",linewidth=1.5,label="Rolling Reward Std")
    axes[1].fill_between(xs,sp,alpha=0.2,color="#ffa657")
axes[1].set_title("Entropy Proxy (Rolling Std)"); axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/reinforce_entropy_curve.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: reinforce_entropy_curve.png")
print("\n All REINFORCE plots done!")

Saved: reinforce_experiments.png
Saved: reinforce_reward_curve.png
Saved: reinforce_entropy_curve.png

 All REINFORCE plots done!


##  Algorithm Comparison And Convergence Plot

In [ ]:
import matplotlib.gridspec as gridspec
np.random.seed(42)
def collect_rewards(model,n=100):
    env=SistaHealthEnv(); rewards=[]
    for _ in range(n):
        obs,_=env.reset(); ep_r=0; done=False
        while not done:
            action,_=model.predict(obs,deterministic=True)
            obs,r,term,trunc,_=env.step(int(action))
            ep_r+=r; done=term or trunc
        rewards.append(ep_r)
    env.close(); return rewards

print("Collecting 100-episode eval data for all models...")
eval_dqn=collect_rewards(dqn_best_model)
eval_ppo=collect_rewards(ppo_best_model)
eval_rf =collect_rewards(rf_best_model)
print("Done!")

# Cumulative + convergence
fig=plt.figure(figsize=(18,12))
fig.suptitle("Cumulative Reward + Convergence — All Algorithms\nSista Health RL",fontsize=14,fontweight="bold")
gs=gridspec.GridSpec(2,3,figure=fig,hspace=0.45,wspace=0.3)
algo_eval=[("DQN",eval_dqn,"#58a6ff"),("PPO",eval_ppo,"#25d366"),("REINFORCE",eval_rf,"#e3b341")]
for idx,(name,rewards,color) in enumerate(algo_eval):
    ax=fig.add_subplot(gs[0,idx])
    cum=np.cumsum(rewards)
    ax.plot(cum,color=color,linewidth=2,label="Cumulative Reward")
    ax.fill_between(range(len(cum)),cum,alpha=0.2,color=color)
    ax.axhline(np.mean(cum[-20:]),color="pink",linestyle="--",linewidth=1.5,alpha=0.8)
    ax.set_title(f"{name}\nFinal: {int(cum[-1])}"); ax.set_xlabel("Episode"); ax.set_ylabel("Cumulative Reward")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax4=fig.add_subplot(gs[1,:])
for name,rewards,color in algo_eval:
    w=10; sm=[np.mean(rewards[max(0,i-w):i+1]) for i in range(len(rewards))]
    ax4.plot(sm,color=color,linewidth=2,label=f"{name} (mean={np.mean(rewards):.1f})")
ax4.axhline(0,color="black",linewidth=0.5,linestyle="--")
ax4.set_title("Convergence — Smoothed Episode Rewards",fontsize=13,fontweight="bold")
ax4.set_xlabel("Episode"); ax4.set_ylabel("Smoothed Reward"); ax4.legend(fontsize=10); ax4.grid(alpha=0.3)
plt.savefig(f"{RESULTS_DIR}/graph1_cumulative_rewards.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: graph1_cumulative_rewards.png")

# Bar comparison
fig,axes=plt.subplots(2,2,figsize=(14,10))
fig.suptitle("Algorithm Comparison — DQN vs PPO vs REINFORCE",fontsize=14,fontweight="bold")
all_results=[("DQN",dqn_results,"#58a6ff"),("PPO",ppo_results,"#25d366"),("REINFORCE",rf_results,"#e3b341")]
ax=axes[0,0]
for nm,res,col in all_results:
    ax.plot([r["Run"] for r in res],[r["Mean Reward"] for r in res],color=col,marker="o",linewidth=2,markersize=5,label=nm,alpha=0.85)
ax.set_title("Mean Reward per Run"); ax.legend(); ax.grid(alpha=0.3)
ax=axes[0,1]
bm=[max(r["Mean Reward"] for r in res) for _,res,_ in all_results]
bs=[min(r["Std Reward"] for r in res)  for _,res,_ in all_results]
nms=[n for n,_,_ in all_results]; cls=[c for _,_,c in all_results]
brs=ax.bar(nms,bm,color=cls,alpha=0.85,yerr=bs,capsize=6)
for bar,val in zip(brs,bm): ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+1,f"{val:.1f}",ha="center",fontweight="bold")
ax.set_title("Best Run Reward"); ax.set_ylabel("Mean Reward")
ax=axes[1,0]
avs=[np.mean([r["Std Reward"] for r in res]) for _,res,_ in all_results]
ax.bar(nms,avs,color=cls,alpha=0.85); ax.set_title("Avg Std Dev (Lower=Stable)"); ax.set_ylabel("Mean Std")
ax=axes[1,1]
all_vals=[[r["Mean Reward"] for r in res] for _,res,_ in all_results]
bp=ax.boxplot(all_vals,labels=nms,patch_artist=True)
for patch,col in zip(bp["boxes"],cls): patch.set_facecolor(col); patch.set_alpha(0.7)
ax.set_title("Reward Distribution"); ax.set_ylabel("Mean Reward"); ax.grid(axis="y",alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/algorithm_comparison.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: algorithm_comparison.png")

# PG Generalization heatmap
fig,axes=plt.subplots(1,2,figsize=(14,6)); fig.suptitle("PG Generalization — All User Profiles",fontsize=13,fontweight="bold")
for ax,(model,name) in zip(axes,[(ppo_best_model,"PPO"),(rf_best_model,"REINFORCE")]):
    env2=SistaHealthEnv(); grid=np.zeros((3,3))
    for lang in range(3):
        for lit in range(3):
            rws=[]
            for _ in range(15):
                ob,_=env2.reset(); ob[0]=lang; ob[3]=lit; ob[4]=0
                ep=0; dn=False; st=0
                while not dn and st<10:
                    ac,_=model.predict(ob,deterministic=True)
                    ob,rw,tr,tc,_=env2.step(int(ac)); ep+=rw; dn=tr or tc; st+=1
                rws.append(ep)
            grid[lit,lang]=np.mean(rws)
    env2.close()
    im=ax.imshow(grid,cmap="RdYlGn",aspect="auto",vmin=-20,vmax=140); plt.colorbar(im,ax=ax)
    ax.set_xticks([0,1,2]); ax.set_xticklabels(["English","Yoruba","Pidgin"])
    ax.set_yticks([0,1,2]); ax.set_yticklabels(["Low","Medium","High"])
    ax.set_title(f"{name} — Reward Heatmap")
    for i in range(3):
        for j in range(3): ax.text(j,i,f"{grid[i,j]:.0f}",ha="center",va="center",fontsize=11,fontweight="bold")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/pg_generalization.png",dpi=150,bbox_inches="tight"); plt.show(); print("Saved: pg_generalization.png")

print("\n"+"="*55)
print(f"  DQN       mean: {np.mean(eval_dqn):.2f} ± {np.std(eval_dqn):.2f}")
print(f"  PPO       mean: {np.mean(eval_ppo):.2f} ± {np.std(eval_ppo):.2f}")
print(f"  REINFORCE mean: {np.mean(eval_rf):.2f} ± {np.std(eval_rf):.2f}")
winner=["DQN","PPO","REINFORCE"][np.argmax([np.mean(eval_dqn),np.mean(eval_ppo),np.mean(eval_rf)])]
print(f"\n   Best algorithm: {winner}")
print("="*55)

Done!
Saved: graph1_cumulative_rewards.png


/tmp/ipykernel_55/3095267935.py:62: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp=ax.boxplot(all_vals,labels=nms,patch_artist=True)


Saved: algorithm_comparison.png
Saved: pg_generalization.png

  DQN       mean: 107.10 ± 29.37
  PPO       mean: 113.10 ± 32.64
  REINFORCE mean: 109.82 ± 33.19

   Best algorithm: PPO


##  Run Best Agent (Video Demo Cell)

In [ ]:
algo_means=[np.mean(eval_dqn),np.mean(eval_ppo),np.mean(eval_rf)]
algo_names=["DQN","PPO","REINFORCE"]
algo_models=[dqn_best_model,ppo_best_model,rf_best_model]
best_idx=int(np.argmax(algo_means))
best_algo=algo_names[best_idx]; best_model=algo_models[best_idx]
print(f"Loading best model: {best_algo}  (mean reward: {algo_means[best_idx]:.2f})")

env=SistaHealthEnv(); N_EVAL=50; all_ep_rewards=[]

for ep in range(N_EVAL):
    obs,info=env.reset(); ep_reward,done,step=0,False,0
    if ep<5:
        print(f"\n{'='*55}")
        print(f"  EPISODE {ep+1} | {info['language']} | {info['domain']} | Literacy: {info['literacy']}")
        print(f"{chr(45)*55}")
    while not done:
        action,_=best_model.predict(obs,deterministic=True)
        obs,reward,term,trunc,info=env.step(int(action))
        ep_reward+=reward
        sign="✅" if reward>0 else ("⚠️" if reward==0 else "❌")
        if ep<5: print(f"  Step {step+1:2d} | {env.ACTIONS[int(action)]:20s} | {reward:+.0f} {sign}")
        step+=1; done=term or trunc
    if ep<5: print(f"  Episode Reward: {ep_reward:.1f}")
    all_ep_rewards.append(ep_reward)

print(f"\n{'='*55}")
print(f"  FINAL SUMMARY — {best_algo}")
print(f"  Episodes evaluated : {N_EVAL}")
print(f"  Mean Reward        : {np.mean(all_ep_rewards):.2f}")
print(f"  Std Reward         : {np.std(all_ep_rewards):.2f}")
print(f"  Min / Max          : {np.min(all_ep_rewards):.1f} / {np.max(all_ep_rewards):.1f}")
print(f"{'='*55}")
env.close()

Loading best model: PPO  (mean reward: 113.10)

  EPISODE 1 | Yoruba | Sexual Health | Literacy: Medium
-------------------------------------------------------
  Step  1 | Voice Note           | +7 ✅
  Step  2 | Voice Note           | +7 ✅
  Step  3 | Voice Note           | +7 ✅
  Step  4 | Voice Note           | +7 ✅
  Step  5 | Voice Note           | +7 ✅
  Step  6 | Voice Note           | +7 ✅
  Step  7 | Voice Note           | +7 ✅
  Step  8 | Voice Note           | +7 ✅
  Step  9 | Voice Note           | +7 ✅
  Step 10 | Voice Note           | +7 ✅
  Episode Reward: 70.0

  EPISODE 2 | English | Maternal Health | Literacy: Low
-------------------------------------------------------
  Step  1 | Voice Note           | +14 ✅
  Step  2 | Voice Note           | +14 ✅
  Step  3 | Voice Note           | +14 ✅
  Step  4 | Voice Note           | +14 ✅
  Step  5 | Voice Note           | +14 ✅
  Step  6 | Voice Note           | +14 ✅
  Step  7 | Voice Note           | +14 ✅
  Step  8 | Voice

##  Scenario Breakdown

In [ ]:
from collections import defaultdict
scenario_rewards=defaultdict(list)
action_counts=defaultdict(lambda: defaultdict(int))
env2=SistaHealthEnv()
for ep in range(100):
    obs,info=env2.reset(); ep_reward=0; done=False
    lang=info["language"]; domain=info["domain"]; lit=info["literacy"]
    while not done:
        action,_=best_model.predict(obs,deterministic=True)
        obs,reward,term,trunc,info=env2.step(int(action))
        ep_reward+=reward
        action_counts[f"{lang}|{lit}"][env2.ACTIONS[int(action)]]+=1
        done=term or trunc
    scenario_rewards[f"{lang} | {domain} | {lit}"].append(ep_reward)
env2.close()
print(f"\n{'='*65}"); print("  BREAKDOWN BY SCENARIO (100 episodes)"); print(f"{'='*65}")
for scenario,rewards in sorted(scenario_rewards.items()):
    print(f"  {scenario:<45} mean={np.mean(rewards):6.1f}  n={len(rewards)}")
print(f"\n{'='*65}"); print("  DOMINANT ACTION BY PROFILE"); print(f"{'='*65}")
for profile,actions in sorted(action_counts.items()):
    total=sum(actions.values()); dominant=max(actions,key=actions.get)
    print(f"  {profile:<25} → {dominant} ({actions[dominant]/total*100:.0f}%)")


  BREAKDOWN BY SCENARIO (100 episodes)
  English | Maternal Health | High              mean= 100.0  n=7
  English | Maternal Health | Low               mean= 140.0  n=2
  English | Maternal Health | Medium            mean=  70.0  n=4
  English | Sexual Health | High                mean= 100.0  n=8
  English | Sexual Health | Low                 mean= 140.0  n=5
  English | Sexual Health | Medium              mean=  70.0  n=7
  Pidgin | Maternal Health | High               mean= 100.0  n=6
  Pidgin | Maternal Health | Low                mean= 160.0  n=8
  Pidgin | Maternal Health | Medium             mean=  90.0  n=4
  Pidgin | Sexual Health | High                 mean= 100.0  n=6
  Pidgin | Sexual Health | Low                  mean= 160.0  n=4
  Pidgin | Sexual Health | Medium               mean=  90.0  n=8
  Yoruba | Maternal Health | High               mean= 100.0  n=4
  Yoruba | Maternal Health | Low                mean= 140.0  n=3
  Yoruba | Maternal Health | Medium             me

##  Output File Listing

In [ ]:
import os
print("All output files in /kaggle/working:\n")
for root,dirs,files in os.walk("/kaggle/working"):
    dirs[:]=[d for d in dirs if d not in ["__pycache__"]]
    level=root.replace("/kaggle/working","").count(os.sep)
    indent="  "*level
    if level==0: print("/kaggle/working/")
    else: print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(files):
        print(f"{"  "*(level+1)}{f}")
print("\n Run complete. Download /kaggle/working via Kaggle output panel.")

All output files in /kaggle/working:

/kaggle/working/
  models/
    dqn/
      best_dqn_model.zip
      best_run.txt
    pg/
      ppo/
        best_ppo_model.zip
        best_run.txt
      reinforce/
        best_reinforce_model.zip
        best_run.txt
  .virtual_documents/
    __notebook_source__.ipynb
  training/
  results/
    algorithm_comparison.png
    dqn_convergence.png
    dqn_experiments.png
    dqn_generalization.png
    dqn_objective_curve.png
    dqn_results.csv
    dqn_reward_curve.png
    graph1_cumulative_rewards.png
    pg_generalization.png
    ppo_entropy_curve.png
    ppo_experiments.png
    ppo_results.csv
    ppo_reward_curve.png
    reinforce_entropy_curve.png
    reinforce_experiments.png
    reinforce_results.csv
    reinforce_reward_curve.png
  environment/
    custom_env.py

 Run complete. Download /kaggle/working via Kaggle output panel.


In [ ]:
import shutil
import os

# Zip everything in /kaggle/working
shutil.make_archive('/kaggle/working/sista_health_results', 'zip', '/kaggle/working')

print("Zip created: sista_health_results.zip")
print(f"Size: {os.path.getsize('/kaggle/working/sista_health_results.zip') / 1024 / 1024:.1f} MB")

Zip created: sista_health_results.zip
Size: 1.7 MB


In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working/sista_health_results.zip')

/kaggle/working/sista_health_results.zip